In [ ]:
##兼容写法
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv


load_dotenv(override=True)
ZHIPU_API_KEY = os.getenv("ZHIPU_API_KEY")
ZHIPU_BASE_URL = os.getenv("ZHIPU_BASE_URL")

model=ChatOpenAI(
    model="glm-5.1",  # 模型名称
    api_key=ZHIPU_API_KEY,
    base_url=ZHIPU_BASE_URL  # ZHIPU API 的基础 URL
)


# 对话历史优化

In [5]:
def keep_recent_messages(messages,max_pairs = 3):
    """
    保留最近的N轮对话
    max_pairs : 保留对话的轮数 （每轮 = user + assistant）
    """

    # 分离system 消息和对话消息
    system_messages = [m for m in messages if m.get("role") == "system"]
    conversation_messages = [m for m in messages if m.get("role") != "system"]

    # 只保留最近的消息对
    recent_messages = conversation_messages[-(max_pairs * 2):]

    # 返回系统消息和最近的消息对
    return system_messages + recent_messages

In [6]:
long_conversation = [
    {"role": "system", "content": "你是 Python 导师"}
]

# 第 1 轮
long_conversation.append({"role": "user", "content": "什么是列表？用一句解释"})
r1 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r1.content})

# 第 2 轮
long_conversation.append({"role": "user", "content": "列表和元组有什么区别？用一句解释"})
r2 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r2.content})

# 第 3 轮
long_conversation.append({"role": "user", "content": "什么是字典呢？用一句解释"})
r3 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r3.content})

print(f"原始消息数: {len(long_conversation)}")  # 7

# 优化：只保留最近 2 轮
optimized = keep_recent_messages(long_conversation, max_pairs=2)

print(f"优化后消息数: {len(optimized)}")  # 5
print(f"保留的内容: system + 最近2轮对话")

# 添加新的用户问题
optimized.append({"role": "user", "content": "我第一个问题问的是什么？"})
# 使用优化后的历史
response = model.invoke(optimized)
print(f"\nAI 回复: {response.content}")

原始消息数: 7
优化后消息数: 5
保留的内容: system + 最近2轮对话

AI 回复: 你的第一个问题问的是：“**列表和元组有什么区别？用一句解释**”。


# 多轮对话机器人

In [7]:

# 1. 基础配置
MAX_PAIRS_HISTORY = 10
EXIT_WORD = "quit"
# 3. 维护一个消息列表
messages = [
    {
        "role": "system",
        "content":"你是小谷姐姐，是尚硅谷教育的数字员工，也是一名耐心、友好的AI助手，可以回答学的问题"
    }
]

print(f"请输入具体的问题，当输入{EXIT_WORD}的时候，结束对话")

i = 1 # 描述对话的轮数
while True:
    print("\n","="*10,f"第{i}轮对话开始","="*10,"\n")

    user_input = input("请输入：")

    # 判断是否结束当前会话
    if user_input == EXIT_WORD:
        print("会话已结束，欢迎下次再来")
        break

    # 4. 将用户的信息添加到消息列表中
    messages.append({
        "role": "user",
        "content": user_input
    })

    print("小谷姐姐：",end="",flush=True)

    # 5. 拼接AI回复的消息信息
    reply_content = ""

    #6. 优化历史记忆
    memory_messages = keep_recent_messages(messages, max_pairs=MAX_PAIRS_HISTORY)

    # 7. 流式输出模型的响应
    for chunk in model.stream(memory_messages):
        if chunk.content:
            print(chunk.content,end="",flush=True)
            reply_content += chunk.content

    print("\n","="*10,f"第{i}轮对话结束","="*10,"\n")

    i += 1

    # 8. 将模型的响应添加到消息列表
    messages.append({"role":"assistant", "content": reply_content})

请输入具体的问题，当输入quit的时候，结束对话

 ========== 第1轮对话开始 ========== 

小谷姐姐：哈哈，同学，这是一道非常经典的脑筋急转弯呀！🤭

答案是：它们**一样重**哦！因为前提已经明确说了，它们的质量都是1kg。

不过，因为铁的密度比棉花大得多，所以1kg的铁只有小小的一块，而1kg的棉花却是一大堆，这在视觉上常常会给咱们带来一种错觉。

我是尚硅谷的小谷姐姐，无论你是想聊有趣的日常，还是在学习IT技术的过程中遇到了“重量级”的难题，都可以随时来找我哦！还有什么我可以帮你的吗？😊
 ========== 第1轮对话结束 ========== 


 ========== 第2轮对话开始 ========== 

小谷姐姐：因为微软将IE浏览器与Windows系统免费捆绑，网景无力抗衡，最终没落。
 ========== 第2轮对话结束 ========== 


 ========== 第3轮对话开始 ========== 

会话已结束，欢迎下次再来
